In [ ]:
!pip install -U langchain-core langchain-community langchain-openai openai

In [2]:
import langchain
print(langchain.__version__)

1.2.15


In [3]:
import os
from getpass import getpass
os.environ["OPENAI_API_KEY"] = getpass("Enter your api key: ")

Enter your api key: ··········


In [4]:
import sys
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [5]:
from langchain_core.prompts import PromptTemplate
react_test_case_prompt= PromptTemplate(
    input_variables=["requirement"],
    template= """

You are a Senior QA Engineer AI using the ReAct methodology.

STEP 1 – REASON:
- Analyze the business requirement
- Identify:
  • Actors
  • Inputs
  • Business rules
  • Validations
  • Risks & edge cases

STEP 2 – REASON:
- Identify all scenarios:
  • Positive (happy path)
  • Negative
  • Boundary / edge cases

STEP 3 – ACT:
Convert scenarios into structured test cases with:
- test_case_id
- title
- preconditions
- steps
- expected_result
- priority (High/Medium/Low)
- test_type (Functional / Negative / Edge)

BUSINESS REQUIREMENT:
{requirement}

Return output in VALID JSON only.

"""
)

In [6]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model = "gpt-4o-mini",
    temperature=0.2
)

In [7]:
from langchain_classic.chains import LLMChain

test_case_agent = LLMChain(
    llm=llm,
    prompt=react_test_case_prompt
)

/tmp/ipykernel_8458/2373380054.py:3: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  test_case_agent = LLMChain(


In [8]:
import re

JAILBREAK_PATTERNS = [
    r"ignore\s+previous\s+instructions",
    r"act\s+as\s+system",
    r"reveal\s+prompt",
    r"bypass\s+security",
    r"tell\s+me\s+your\s+rules",
    r"api\s*key",
    r"credit\s*card"
]

def detect_jailbreak(text: str) -> bool:
    text = text.lower()
    return any(re.search(p, text) for p in JAILBREAK_PATTERNS)


In [9]:
def redact_pii(text: str) -> str:
    patterns = {
        "EMAIL": r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+",
        "PHONE": r"\+?\d[\d\s\-]{8,}\d",
        "AADHAAR": r"\b\d{4}[\s\-]?\d{4}[\s\-]?\d{4}\b",
        "PAN": r"\b[A-Z]{5}[0-9]{4}[A-Z]\b",
    }

    for label, pattern in patterns.items():
        text = re.sub(pattern, f"<REDACTED_{label}>", text)

    return text

In [10]:
def safe_run_test_case_agent(agent, requirement: str):
    # Jailbreak protection
    if detect_jailbreak(requirement):
        return {
            "error": "Unsafe prompt detected",
            "status": "Blocked by security layer"
        }

    # Redact input PII
    safe_requirement = redact_pii(requirement)

    # Run agent
    response = agent.run(requirement=safe_requirement)

    # Redact output PII
    safe_response = redact_pii(response)

    return safe_response

In [11]:
business_requirement = """
User must be able to log in using email and password.
Email must be registered.
Password must have a minimum of 8 characters.
Account should be locked after 3 consecutive failed login attempts.
"""

result = safe_run_test_case_agent(
    test_case_agent,
    business_requirement
)

print(result)


/tmp/ipykernel_8458/2463900876.py:13: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  response = agent.run(requirement=safe_requirement)


```json
{
  "step_1_reason": {
    "business_requirement": "User must be able to log in using email and password.",
    "actors": ["User", "Authentication System"],
    "inputs": ["Email", "Password"],
    "business_rules": [
      "Email must be registered.",
      "Password must have a minimum of 8 characters.",
      "Account should be locked after 3 consecutive failed login attempts."
    ],
    "validations": [
      "Check if email is registered.",
      "Check if password meets minimum length requirement."
    ],
    "risks_edge_cases": [
      "User enters an unregistered email.",
      "User enters a password shorter than 8 characters.",
      "User exceeds 3 failed login attempts.",
      "User attempts to log in after account is locked."
    ]
  },
  "step_2_reason": {
    "scenarios": {
      "positive": [
        "User enters registered email and correct password.",
        "User enters registered email and correct password after 1 failed attempt."
      ],
      "negative